# S1. 생성 영상의 의미 재분류와 검증

**“만들어 달라고 한 장면”과 “실제로 만들어진 장면”을 구분하는 실습입니다.**

예를 들어 “비 오는 밤에 옆 차로 차량이 끼어든다”라고 요청해도, 생성 결과에는 비와 야간 조명만 있고 끼어드는 행동이 없을 수 있습니다. 이 노트북에서는 목표를 그대로 정답으로 쓰지 않고, 관측한 조건을 따로 기록한 뒤 비교합니다.

처음 실행할 때는 **Kernel → Restart Kernel and Run All Cells**로 전체 흐름을 확인하세요. 기본 설정은 CPU 예제만 실행하며, 영상 다운로드·GPU 모델 로딩·유료 생성 작업은 시작하지 않습니다.

- [기술 설명과 실행 안내](../docs/ko/SEMANTIC_VIDEO_GUIDE.md)
- [권장 판별 조건과 반례](../docs/ko/SEMANTIC_CRITERIA.md)
- [검증 기록과 미실행 범위](../docs/ko/SEMANTIC_VALIDATION.md)

CPU 예제의 관측값과 확률은 사람이 만든 학습용 자료입니다. 이 출력으로 실제 모델의 정확도나 속도를 평가할 수는 없습니다.

## 1. 이번 실습에서 배우는 흐름

**생성 영상 → 사건 구간 선택 → 시간순 프레임 → 관측 라벨 → 목표 비교 → 사람 검수 → 필요하면 생성 조건 수정**

기존 AV 3.0 저장소의 M4(Transfer), M5(Predict)가 만든 `manifest.json`을 입력으로 사용할 수 있습니다. 원격 개편 버전의 M5(Transfer), M6(Predict)도 같은 필드 계약으로 읽습니다. 이 S1 노트북은 기존 M 번호를 바꾸지 않는 추가 실습입니다.

이번 단계는 영상의 픽셀을 바꾸는 기능이 아닙니다. 의미 라벨과 검수 근거를 정리하고, 어떤 조건을 다시 만들어야 할지 제안합니다. 물리적으로 올바른 움직임이나 주행 정책의 안전성은 후속 검증이 필요합니다.

In [ ]:
# 현재 위치가 저장소 루트이든 notebooks 폴더이든 공통 코드를 찾습니다.
from pathlib import Path
import sys
import json
import copy
import tempfile
import csv
from collections import Counter
from IPython.display import display, Markdown

candidates = [Path.cwd(), *Path.cwd().parents]
REPO = next((p for p in candidates if (p / "src/av_decision/semantic.py").is_file()), None)
if REPO is None:
    raise RuntimeError("이 노트북을 Decision 프로젝트 폴더 안에서 열어 주세요.")
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

from av_decision.semantic import (
    load_rubric, demo_data, FixtureProvider, run_batch, regeneration_plan,
    import_manifest, read_json, write_json, evaluate_human_labels,
)

# 매번 새 작업 폴더를 만들어 이전 실행 결과를 덮어쓰지 않습니다.
RUN_ROOT = Path(tempfile.mkdtemp(prefix="av-semantic-study-"))
rubric = load_rubric()
print("사용 중인 Python:", sys.executable)
print("저장소:", REPO)
print("이번 실행 폴더:", RUN_ROOT)
print("판별 기준 버전:", rubric["rubric_id"])

# 표에 들어갈 문자는 이스케이프하여 원래 내용을 보여 줍니다.
def show_table(headers, rows):
    def safe(value):
        return str(value).replace("|", "&#124;").replace("\n", " ").replace("<", "&lt;")
    lines = ["| " + " | ".join(map(safe, headers)) + " |",
             "| " + " | ".join(["---"] * len(headers)) + " |"]
    lines += ["| " + " | ".join(map(safe, row)) + " |" for row in rows]
    display(Markdown("\n".join(lines)))

## 2. 긴 문장을 작은 판별 조건으로 나눕니다

날씨, 조명, 사건, 자료 품질을 따로 판단합니다. 모든 영상에 동일한 질문을 보내며 **생성 목표와 원래 파일명은 모델 입력에서 제외**합니다. 모델이 생성 의도를 보고 답을 맞추는 편향을 줄이기 위한 구조입니다.

사건에서 `no`는 필요한 장면을 확인했지만 사건이 없다는 뜻이고, `unknown`은 있었는지 없었는지 확인할 수 없다는 뜻입니다. 보이지 않은 사건을 무조건 `no`로 만들지 않는 것이 중요합니다.

In [ ]:
show_table(
    ["기준 ID", "질문", "선택지", "시간 변화 필요"],
    [(key, q["title"], ", ".join(q["criteria"]), "예" if q["temporal"] else "아니요")
     for key, q in rubric["questions"].items()],
)
print("기본 임계값:", json.dumps(rubric["thresholds"], ensure_ascii=False))

확률 0.80 이상, 1위와 2위의 차이 0.20 이상, 관측 공백 0.30초 이하는 **실습을 시작하기 위한 값**입니다. 검증된 운영 기준이나 모델 정확도를 뜻하지 않습니다. 실제 자료에서는 validation 정답과 보류율을 함께 보며 조정해야 합니다.

`weather=clear`에는 흐린 날도 포함됩니다. “흐림”을 별도로 학습하려면 기준부터 확장해야 합니다. `continuity=consistent`도 샘플에서 명백한 결함을 찾지 못했다는 뜻이며, 물리적으로 타당하다는 보장은 아닙니다.

## 3. GPU 없이 여섯 가지 결과를 만들어 봅니다

다음 셀은 영상 대신 **가상 관측값**을 사용합니다. 목표 일치, 사건 불일치, 가림, 생성 결함, 부족한 원본 시간 정보, 어려워도 유효한 장면을 각각 넣었습니다.

미리 만들어 둔 확률 0.94도 실습용 숫자입니다. 실제 모델이 이 정도로 정확하다는 뜻이 아닙니다.

In [ ]:
records, fixture_answers = demo_data(rubric)
demo_report = run_batch(records, FixtureProvider(fixture_answers), rubric, RUN_ROOT / "demo")
status_names = {
    "candidate": "학습 후보", "relabel_candidate": "재분류 후보",
    "review": "판단 보류", "quarantine": "생성 결함 검토",
}
show_table(["처리 상태", "영상 수"],
           [(status_names[k], v) for k, v in demo_report["counts"].items()])
show_table(["예제", "결과", "불일치", "보류 이유", "어려운 장면"], [
    [r["clip_id"], status_names[r["disposition"]],
     ", ".join(r["mismatched_criteria"]) or "없음",
     ", ".join(r["review_reasons"]) or "없음", r["difficult_scene"]]
    for r in demo_report["results"]
])

결과는 **학습 후보 2개, 재분류 후보 1개, 보류 2개, 결함 검토 1개**가 됩니다.

- 가림으로 사건을 확인하지 못한 영상은 `unknown`과 함께 보류합니다.
- 원본이 2Hz이면 파일을 10fps로 저장했더라도 시간 사건 판정은 보류합니다.
- 안개나 어둠 때문에 `visibility=limited`여도 사건 근거가 충분하면 후보로 남깁니다.
- 모든 후보는 사람 검수 전입니다. `requires_human_approval=true`를 확인하세요.

`observed`에는 전체 질문의 답이 들어가지만, 목표와 공통 품질 조건으로 검사한 항목은 `checked_criteria`입니다. 다른 부가 라벨까지 검증이 끝났다고 해석하지 마세요.

In [ ]:
# 요청은 끼어들기였지만, 관측은 끼어들기 없음인 예제를 자세히 봅니다.
mismatch = demo_report["results"][1]
show_table(["조건", "생성 목표", "모델 관측", "선택 확률", "1·2위 차이"], [
    [key, target, mismatch["observed"][key]["choice"],
     round(mismatch["observed"][key]["confidence"], 3),
     round(mismatch["observed"][key]["margin"], 3)]
    for key, target in mismatch["target"].items()
])
print("검사한 기준:", mismatch["checked_criteria"])
print("사람 검수 필요:", mismatch["requires_human_approval"])

## 4. 임계값을 올리면 어떤 일이 생길까요?

확률 기준을 0.99로 올려 보겠습니다. 이 예제에서는 모델 역할을 하는 가상 관측이 모두 0.94이므로 보류가 늘어납니다. 기준을 높이면 정답만 남는 것이 아니라, 맞는 자료도 함께 보류될 수 있다는 점을 확인하는 실험입니다.

실제 모델 비교에서는 정확도와 보류율을 함께 봐야 합니다. 임계값을 고르는 데 최종 test 정답을 반복해서 사용하지 마세요.

In [ ]:
strict_rubric = copy.deepcopy(rubric)
strict_rubric["rubric_id"] = "av-semantic-ko-v1-strict-demo"
strict_rubric["thresholds"]["min_probability"] = 0.99
strict_report = run_batch(records, FixtureProvider(fixture_answers), strict_rubric, RUN_ROOT / "strict-demo")
show_table(["상태", "기본 기준", "확률 0.99 기준"], [
    [status_names[key], demo_report["counts"][key], strict_report["counts"][key]]
    for key in status_names
])

## 5. 재생성은 실행하지 않고 수정 제안부터 만듭니다

현재 코드는 `relabel_candidate`에 대해 기준별 수정 문장을 저장합니다. 원래 생성 1회를 포함한 최대 횟수는 2회입니다. `test`와 `unassigned` 자료는 피드백 생성 대상으로 삼지 않습니다.

수정 제안은 원래 프롬프트·제어 조건과 함께 검토하세요. 재생성 후에는 새로운 `clip_id`와 증가한 `attempt`를 쓰되, 원본 `source_group`과 `split`은 유지합니다. validation 파생 영상을 train으로 옮기면 안 됩니다.

In [ ]:
plan = regeneration_plan(records[1], mismatch, rubric)
print(json.dumps(plan, ensure_ascii=False, indent=2))

# 같은 관측이라도 시험 자료이거나 예산을 썼다면 제안을 보류합니다.
held_test = regeneration_plan(dict(records[1], split="test"), mismatch, rubric)
held_budget = regeneration_plan(dict(records[1], attempt=2), mismatch, rubric)
print("시험 자료의 상태:", held_test["status"])
print("2회 생성한 자료의 상태:", held_budget["status"])

## 6. 사람이 작성한 정답과 비교합니다

실제 검수는 `human_review.csv`를 복사한 뒤 사람이 영상을 보고 `human_choice`, `reviewer`, `note`를 채우는 방식입니다. 가능하면 모델 답을 보기 전에 독립적으로 검수하세요.

아래에서는 계산법을 배우기 위해 **가상 정답 세 행만** 넣습니다. 이 숫자는 모델 성능이 아닙니다. 양성 정답 중 `unknown`인 경우를 단순한 음성 답과 구분해서 세는지 살펴보세요.

In [ ]:
# 계산 방법만 보여 주는 가상 정답입니다. 실제 영상 정답으로 사용하지 않습니다.
example_labels = [
    {"clip_id": records[0]["clip_id"], "criterion": "cut_in", "human_choice": "yes"},
    {"clip_id": records[1]["clip_id"], "criterion": "cut_in", "human_choice": "no"},
    {"clip_id": records[2]["clip_id"], "criterion": "cut_in", "human_choice": "yes"},
]
example_metrics = evaluate_human_labels(demo_report["results"], example_labels, rubric)
print("가상 세 행에 대한 계산 예시:")
print(json.dumps(example_metrics, ensure_ascii=False, indent=2))

이진 사건의 `false_negative`는 실제 양성인데 `no`라고 한 수입니다. 실제 양성인데 `unknown`이거나 처리에 실패한 경우는 `unknown_on_positive`에 따로 들어갑니다. 따라서 미탐률만 낮다고 좋아진 것은 아닙니다. 둘을 함께 보고, 전체 보류율과 검수 완료율도 확인하세요.

날씨처럼 다중 선택지인 항목은 현재 함수에서 항목별 정확도를 확인할 수 있습니다. 이진 사건용 오탐·미탐 항목을 날씨의 품질 지표로 해석하지 마세요.

## 7. 실제 생성 결과를 가져올 준비

여기부터는 선택 실습입니다. 우선 `IMPORT_REAL=True`로 바꾸고 manifest·영상 폴더·원본 주행 그룹을 설정합니다. **파일을 읽는 단계와 모델을 실행하는 단계를 나눠 두었습니다.**

- M4/M5를 실행한 서버라면 `MANIFEST_PATH`는 해당 노트북의 `manifest_path`, `VIDEO_ROOT`는 `OUTPUT_DIR`에 해당합니다.
- 다른 서버에서 실행한다면 생성 출력 폴더를 먼저 복사하고 상대 폴더 구조를 유지하세요.
- `source_group`은 실험 이름이 아니라 원본 주행을 가리켜야 합니다.
- 변환된 `source_observation_hz`는 기본 `None`입니다. 원본의 독립 관측 주기를 확인한 뒤 입력하세요.
- `near_collision`, `sudden_brake`는 영상만으로 주장하기 어려워 자동 목표 매핑에서 제외했습니다. 목표 검토가 필요합니다.

In [ ]:
# 자신의 파일과 모델이 준비된 뒤에만 값을 바꿔 주세요.
IMPORT_REAL = False
RUN_REAL = False
MANIFEST_PATH = Path("/data/cosmos-run/manifest.json")
VIDEO_ROOT = Path("/data/cosmos-run")
SOURCE_GROUP = ""                         # 예: 원본 nuScenes log 식별자
SPLIT = "unassigned"                       # 원본 주행 단위로 먼저 분할합니다.

# 모델 경로는 실제 반입한 snapshot을 가리켜야 합니다.
PROVIDER = "valen"                         # valen 또는 cosmos-http
MODEL_DIR = Path("/models/Valen-2B-reviewed-snapshot")
MODEL_REVISION = ""                       # 실제 모델 commit 또는 관리 중인 revision
COSMOS_ENDPOINT = "http://127.0.0.1:8000/v1"
COSMOS_MODEL = ""                         # 서버가 제공하는 실제 모델 이름

# 필요한 영상에만 명시적으로 값을 추가합니다. 아래는 형식 설명입니다.
# RECORD_OVERRIDES = {
#     "실제 clip_id": {
#         "window": {"start_s": 0.0, "end_s": 5.0},
#         "source_observation_hz": 2.0,  # 원본이 2Hz라면 그대로 적습니다.
#         "target": {"weather": "rain", "cut_in": "yes"},
#         "target_needs_review": False, # 목표를 직접 검토했을 때만 바꿉니다.
#     }
# }
RECORD_OVERRIDES = {}

real_records = []
if IMPORT_REAL:
    if not SOURCE_GROUP.strip():
        raise ValueError("SOURCE_GROUP에 원본 주행 식별자를 입력해 주세요.")
    real_records = import_manifest(MANIFEST_PATH, VIDEO_ROOT, rubric, SOURCE_GROUP, SPLIT)
    show_table(["clip_id", "요청 이름", "목표", "목표 검토 필요"], [
        [r["clip_id"], r["requested_name"], r["target"], r["target_needs_review"]]
        for r in real_records
    ])
else:
    print("실제 입력 읽기를 건너뛰었습니다. CPU 실습 결과는 그대로 확인할 수 있습니다.")

표에 나온 ID를 보고 위의 `RECORD_OVERRIDES`에 구간과 원본 주기를 입력한 뒤 셀을 다시 실행하세요. 알 수 없는 목표는 이름만 보고 추측하지 않습니다.

예를 들어 “충돌 직전”이라는 요구를 “선행 차량과 간격이 줄어드는 시각적 단서”로 바꿔 조사한다면 `gap_closing=yes`를 명시적으로 선택할 수 있습니다. 다만 이것은 질문 자체를 좁힌 것이며, 충돌 직전 여부를 검증한 결과가 아닙니다.

기본 4fps·최대 32장은 비교를 시작할 설정입니다. 8초보다 긴 영상은 사건 전후를 담은 window를 지정하세요. 실제 영상 처리에는 `pyproject.toml`의 `video` 선택 패키지가 필요합니다.

In [ ]:
# 검토한 변경만 적용합니다. 모델 호출은 다음 셀에서만 수행합니다.
if real_records:
    known_ids = {r["clip_id"] for r in real_records}
    unknown_ids = set(RECORD_OVERRIDES) - known_ids
    if unknown_ids:
        raise ValueError(f"입력 목록에 없는 수정 ID입니다: {sorted(unknown_ids)}")
    for record in real_records:
        record.update(copy.deepcopy(RECORD_OVERRIDES.get(record["clip_id"], {})))
    show_table(["clip_id", "원본 주기", "분석 구간", "목표"], [
        [r["clip_id"], r["source_observation_hz"], r["window"], r["target"]]
        for r in real_records
    ])
else:
    print("불러온 실제 영상이 없습니다.")

## 8. 실제 모델로 관측 라벨을 만듭니다

**Valen:** 로컬에 반입한 모델과 전용 Python/GPU 환경이 필요합니다. 공식 모델의 사용자 정의 추론 코드를 실행하므로 검토한 snapshot을 사용하세요. 실제 가중치의 정밀도 경로를 유지하며, 모든 가중치를 임의로 BF16으로 바꾸지 않습니다.

**Cosmos HTTP:** 멀티이미지 입력을 지원하는 별도 서버가 필요합니다. 이미지가 지정 서버로 전송됩니다. 자유 생성 응답의 숫자를 보정된 확률로 사용하지 않으므로, 현재 구현에서는 모두 사람 검수 대상으로 남습니다. 키가 필요하면 환경변수 `SEMANTIC_VLM_API_KEY`를 사용하세요.

두 경로는 동일한 방식으로 추출한 시간순 이미지를 입력합니다. 모델별 네이티브 비디오 성능을 재현하는 실험은 아닙니다. 실제 모델 설치와 서버 설정은 [기술 설명서](../docs/ko/SEMANTIC_VIDEO_GUIDE.md)를 참고하세요.

In [ ]:
real_report = None
if RUN_REAL:
    if not real_records:
        raise ValueError("먼저 IMPORT_REAL=True로 입력을 불러오고 검토해 주세요.")
    if not MODEL_REVISION.strip():
        raise ValueError("실제로 사용하는 모델 revision을 입력해 주세요.")
    from av_decision.semantic_providers import ValenProvider, CosmosHTTPProvider
    sampling = {"sample_fps": 4.0, "max_frames": 32}
    if PROVIDER == "valen":
        provider = ValenProvider(MODEL_DIR, MODEL_REVISION, device="cuda", **sampling)
    elif PROVIDER == "cosmos-http":
        provider = CosmosHTTPProvider(COSMOS_ENDPOINT, COSMOS_MODEL, MODEL_REVISION, **sampling)
    else:
        raise ValueError("PROVIDER는 valen 또는 cosmos-http여야 합니다.")
    real_output = RUN_ROOT / f"real-{PROVIDER}"
    real_report = run_batch(real_records, provider, rubric, real_output)
    print("실제 영상 처리 결과:", real_report["counts"])
    print("검수 폴더:", real_output)
    errors = [(r["clip_id"], r["error"]) for r in real_report["results"] if "error" in r]
    if errors:
        show_table(["처리 실패 영상", "원인"], errors)
else:
    print("실제 모델 실행을 건너뛰었습니다. 모델 성능은 아직 측정하지 않았습니다.")

영상별 오류는 `review`로 기록됩니다. 출력이 생겼다고 모든 영상이 추론에 성공한 것은 아닙니다. `error`와 `review_reasons`를 먼저 확인하세요. 명령행에서는 일부 실패가 있으면 종료 코드 2를 반환합니다.

시간 관련 항목은 원본 관측 주기를 확인하지 못하면 보류합니다. 이때 보류를 없애려고 추정한 fps를 적는 대신, 원본 근거를 확보하거나 그 한계를 보고서에 남겨야 합니다.

In [ ]:
# 결과를 검토할 때 실제로 사용한 프레임 시각과 해시를 함께 확인합니다.
if real_report is not None:
    for result in real_report["results"][:3]:
        evidence = result.get("evidence", {})
        print("영상:", result["clip_id"])
        print("처리:", result["disposition"])
        print("실제 샘플 시각:", evidence.get("sampled_timestamps_s"))
        print("가장 긴 관측 공백:", evidence.get("max_gap_s"))
        print("원본 파일 해시:", evidence.get("video_sha256"))
        print("보류 이유:", result.get("review_reasons", result.get("error")))
else:
    print("실제 영상 근거는 실제 모델 실행 후 확인할 수 있습니다.")

## 9. 실제 사람이 검수한 CSV를 평가합니다

`human_review.csv`의 복사본에 사람이 정답을 채운 뒤 아래 경로를 설정합니다. 값은 `yes`, `no`, `unknown`, `rain`처럼 기준 파일의 선택지 ID를 씁니다. 빈 행은 아직 검수가 끝나지 않은 것으로 보고 평가에서 제외합니다.

평가 함수는 CSV에 적힌 정답에 대해서만 계산합니다. 전체 대상 수와 검수 완료 수를 함께 보고, 처리 실패 영상도 가능하면 원본을 보고 정답을 남겨야 합니다.

In [ ]:
HUMAN_LABEL_CSV = None                     # 예: Path("/data/review/human-reviewed.csv")
if HUMAN_LABEL_CSV is not None:
    if real_report is None:
        raise ValueError("이 CSV와 대응되는 실제 보고서를 먼저 생성하거나 read_json으로 불러오세요.")
    with Path(HUMAN_LABEL_CSV).open(encoding="utf-8-sig", newline="") as stream:
        csv_rows = list(csv.DictReader(stream))
    labels = [row for row in csv_rows if row.get("human_choice", "").strip()]
    metrics = evaluate_human_labels(real_report["results"], labels, rubric)
    expected = len(real_report["results"]) * len(rubric["questions"])
    print(f"항목별 검수 완료: {len(labels)} / {expected}")
    print(json.dumps(metrics, ensure_ascii=False, indent=2))
else:
    print("사람 검수 CSV 평가를 건너뛰었습니다.")

## 10. 결과를 보관하고 다음 실험을 설계합니다

각 실행 폴더에는 입력 사본, 보고서, 사람 검수 CSV, 재생성 수정 제안이 들어갑니다. 실제 영상 경로에서는 프레임·시각·원본 해시도 남습니다. CPU 예제에는 실제 영상 근거 파일이 없습니다.

기본 작업 폴더는 임시 폴더입니다. 제출하거나 장기 보관할 결과는 아래 출력 위치의 **폴더 전체**를 스터디 자료 위치로 복사하세요. 임시 경로가 담긴 보고서는 폴더를 옮긴 후 경로를 다시 확인해야 합니다.

실제 속도를 비교할 때 `elapsed_s`에는 디코딩·최초 모델 로드·추론이 섞여 있다는 점을 기억하세요. 같은 장비·프레임 수·해상도에서 첫 실행과 준비된 모델의 실행을 구분하고, 보류율과 메모리 사용량도 함께 측정하는 것이 좋습니다.

In [ ]:
print("이번 실습의 전체 결과 폴더:", RUN_ROOT)
print("CPU 보고서:", RUN_ROOT / "demo/report.json")
print("사람 검수 양식:", RUN_ROOT / "demo/human_review.csv")
print("재생성 수정 제안:", RUN_ROOT / "demo/regeneration_plans.json")
if real_report is not None:
    print("실제 모델 결과:", real_output)
print("실제 모델을 실행했나요?", real_report is not None)

## 마무리 과제

1. “비 오는 밤에 차량이 끼어든다”는 목표에서 날씨·조명·사건이 각각 틀린 사례를 구분해 보세요.
2. 가림 때문에 판단하지 못한 장면을 `no`로 만들면 학습에 어떤 문제가 생기는지 설명해 보세요.
3. 안개 장면의 보류율이 높다면 자료를 버리기 전에 무엇을 확인해야 할까요?
4. `gap_closing=yes`를 “충돌 직전”의 정답으로 쓰려면 어떤 추가 자료가 필요할까요?
5. 재분류 전후 자료로 같은 모델을 학습하는 대조 실험을 설계해 보세요. 원본 주행 분할과 평가 조건은 고정해야 합니다.

최종 제출물에는 **기준 버전, 실제/가상 실행 여부, 검수 범위, 보류·실패 수, 원본 출처, 아직 확인하지 못한 주장**을 함께 적어 주세요. 작은 분류 모델의 장점은 처리 속도만으로 판단하지 않고, 필요한 기준에서 얼마나 정확하고 일관되게 작동하는지 측정해서 확인합니다.